# DocSight RAG — Data Exploration
Explore SEC filings and PubMed chunks after ingestion.

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import pandas as pd
import matplotlib.pyplot as plt
import config

## Finance Chunks

In [ ]:
finance_chunks = []
with open(config.FINANCE_CHUNKS_PATH) as f:
    for line in f:
        finance_chunks.append(json.loads(line))

df_fin = pd.DataFrame([{
    'ticker': c['metadata'].get('ticker'),
    'type': c['metadata'].get('type'),
    'len': len(c['text'])
} for c in finance_chunks])

print(f'Total finance chunks: {len(df_fin)}')
df_fin.groupby('ticker').size().plot(kind='bar', title='Chunks per Ticker')
plt.tight_layout()
plt.show()

In [ ]:
print('Chunk length distribution:')
df_fin['len'].describe()

In [ ]:
# Preview a chunk
sample = finance_chunks[42]
print('Metadata:', sample['metadata'])
print('\nText preview:')
print(sample['text'][:500])

## Medical Chunks

In [ ]:
medical_chunks = []
with open(config.MEDICAL_CHUNKS_PATH) as f:
    for line in f:
        medical_chunks.append(json.loads(line))

df_med = pd.DataFrame([{
    'source': c['metadata'].get('source', '')[:30],
    'type': c['metadata'].get('type'),
    'year': c['metadata'].get('year', ''),
    'len': len(c['text'])
} for c in medical_chunks])

print(f'Total medical chunks: {len(df_med)}')
df_med['type'].value_counts().plot(kind='pie', title='Chunk Types', autopct='%1.1f%%')
plt.show()

## Test FAISS Retrieval

In [ ]:
from src.data_ingestion.embedder import load_finance_index, load_medical_index

finance_index = load_finance_index()
results = finance_index.similarity_search('Apple revenue 2023', k=3)
for i, doc in enumerate(results):
    print(f'\n--- Result {i+1} ---')
    print('Metadata:', doc.metadata)
    print(doc.page_content[:300])

In [ ]:
medical_index = load_medical_index()
results = medical_index.similarity_search('type 2 diabetes metformin treatment', k=3)
for i, doc in enumerate(results):
    print(f'\n--- Result {i+1} ---')
    print('Metadata:', doc.metadata)
    print(doc.page_content[:300])